In [1]:
!pip install yfinance
import yfinance as yf

zsh:1: /Users/henrikengdal/Documents/GitHub/IND310-Project/.conda/bin/pip: bad interpreter: /Users/henrikengdal/Documents/Skole/IND310/.conda/bin/python3.11: no such file or directory

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip3.12 install --upgrade pip


In [24]:
tick = "EQNR.OL"
data = yf.download(tick)
data.head()

/var/folders/cb/h4grq88s6sjgm0cnxyzjm7xw0000gn/T/ipykernel_74821/3029344475.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tick)
[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,EQNR.OL,EQNR.OL,EQNR.OL,EQNR.OL,EQNR.OL
Date,,,,,
2025-09-29,249.699997,254.500000,249.300003,253.100006,2535940
2025-09-30,243.399994,247.800003,243.399994,246.399994,2549447
2025-10-01,246.000000,247.000000,243.300003,243.399994,3722075
2025-10-02,246.399994,246.899994,242.600006,244.100006,2114474
2025-10-03,247.600006,247.600006,243.899994,244.000000,1740131


In [47]:
data = yf.Ticker("EQNR.OL").history()
data[["Close", "Dividends","Stock Splits"]].head()

,Close,Dividends,Stock Splits
Date,,,
2025-09-29 00:00:00+02:00,249.699997,0.0,0.0
2025-09-30 00:00:00+02:00,243.399994,0.0,0.0
2025-10-01 00:00:00+02:00,246.000000,0.0,0.0
2025-10-02 00:00:00+02:00,246.399994,0.0,0.0
2025-10-03 00:00:00+02:00,247.600006,0.0,0.0


In [ ]:
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
import yfinance as yf

# Define tickers and date range
TICKERS = ["EQNR.OL", "DNB.OL", "AKRBP.OL", "ORK.OL", "MOWI.OL"]
START = "2020-10-15"
END = "2025-10-15"

# Download data for all tickers WITH actions (dividends and splits)
all_data = {}
for ticker in TICKERS:
    print(f"Downloading data for {ticker}...")
    # Use yf.Ticker().history() to ensure we get dividends and stock splits
    ticker_obj = yf.Ticker(ticker)
    all_data[ticker] = ticker_obj.history(start=START, end=END, actions=True, auto_adjust=False)

# Create a combined DataFrame with closing prices
close_prices = pd.DataFrame()
for ticker in TICKERS:
    if not all_data[ticker].empty:
        close_prices[ticker] = all_data[ticker]['Close']

# Remove any rows with NaN values
close_prices = close_prices.dropna()

print(f"Data shape: {close_prices.shape}")
print(f"Columns in data: {list(all_data[TICKERS[0]].columns)}")
close_prices.head()

Data shape: (1257, 5)
Columns in data: ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'Dividends', 'Stock Splits']


,EQNR.OL,DNB.OL,AKRBP.OL,ORK.OL,MOWI.OL
Date,,,,,
2020-10-15 00:00:00+02:00,132.199997,132.750000,149.050003,92.080002,168.250000
2020-10-16 00:00:00+02:00,133.850006,134.750000,152.000000,92.940002,168.250000
2020-10-19 00:00:00+02:00,133.550003,138.300003,152.000000,92.860001,170.350006
2020-10-20 00:00:00+02:00,132.100006,138.500000,150.000000,92.900002,169.899994
2020-10-21 00:00:00+02:00,131.600006,140.000000,150.000000,92.199997,164.699997


In [3]:
# Create interactive line plot with Plotly
fig = go.Figure()

# Define colors for each ticker
colors = ['#2f4d8e', '#ffa659', '#ff6464', '#79ffbc', '#c180ff']

for i, ticker in enumerate(TICKERS):
    if ticker in close_prices.columns:
        fig.add_trace(go.Scatter(
            x=close_prices.index,
            y=close_prices[ticker],
            mode='lines',
            name=ticker,
            line=dict(color=colors[i], width=2)
        ))

# Update layout
fig.update_layout(
    title='Aksjepris Sammenligning - Norske Selskaper',
    xaxis_title='Dato',
    yaxis_title='Aksjepris (NOK)',
    hovermode='x unified',
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01
    ),
    height=600,
    width=1000
)

# Show the plot
fig.show()

In [9]:
# Calculate Adjusted Close Prices Manually
# Formula: Adj_Close = Close * (Cumulative Split Adjustment) * (Cumulative Dividend Adjustment)

def calculate_adjusted_close(ticker_data):
    """
    Manually calculate adjusted close prices using the standard method
    """
    df = ticker_data.copy()
    
    # Get the close, dividends, and stock splits
    close = df['Close'].copy()
    dividends = df['Dividends'].fillna(0)
    splits = df['Stock Splits'].fillna(1)  # Default to 1 (no split)
    
    # Replace 0 splits with 1 (no split)
    splits = splits.replace(0, 1)
    
    # Calculate cumulative split adjustment factor (working backwards from latest date)
    cumulative_split_adjustment = splits[::-1].cumprod()[::-1]
    
    # Calculate dividend adjustment factor (working backwards)
    # This is the standard approach used by financial data providers
    dividend_ratio = pd.Series(1.0, index=close.index)
    
    # Work backwards from the most recent date
    for i in range(len(close) - 1, -1, -1):
        if dividends.iloc[i] > 0:
            # Calculate the ratio for dividend adjustment
            ratio = (close.iloc[i] - dividends.iloc[i]) / close.iloc[i]
            # Apply this ratio to all previous dates
            dividend_ratio.iloc[:i] *= ratio
    
    # Calculate adjusted close
    adjusted_close = close * cumulative_split_adjustment * dividend_ratio
    
    return adjusted_close

# Calculate adjusted close for all tickers
adjusted_close_prices = pd.DataFrame()
print("Calculating manually adjusted close prices...")

for ticker in TICKERS:
    if ticker in all_data and not all_data[ticker].empty:
        print(f"Processing {ticker}...")
        adj_close = calculate_adjusted_close(all_data[ticker])
        adjusted_close_prices[ticker] = adj_close

# Remove any rows with NaN values
adjusted_close_prices = adjusted_close_prices.dropna()

print(f"\nAdjusted Close Data Shape: {adjusted_close_prices.shape}")
print("\nFirst 5 rows of manually calculated adjusted close prices:")
print(adjusted_close_prices.head())

print("\nLast 5 rows of manually calculated adjusted close prices:")
print(adjusted_close_prices.tail())

# Simple comparison with yfinance's built-in adjusted close
print("\n" + "="*60)
print("VALIDATION: Comparing with yfinance's Adj Close")
print("="*60)

sample_ticker = "EQNR.OL"
if sample_ticker in adjusted_close_prices.columns:
    # Get yfinance's adjusted close for this ticker
    yf_adj_close = all_data[sample_ticker]['Adj Close']
    manual_adj_close = adjusted_close_prices[sample_ticker]
    
    print(f"\nComparison for {sample_ticker} (Last 5 days):")
    print("Manual Adj Close:")
    print(manual_adj_close.tail())
    print("\nYFinance Adj Close:")
    print(yf_adj_close.tail())
    
    # Calculate percentage difference
    recent_manual = manual_adj_close.iloc[-1]
    recent_yf = yf_adj_close.iloc[-1]
    diff_percent = ((recent_manual - recent_yf) / recent_yf * 100)
    
    print(f"\nMost recent values:")
    print(f"Manual: {recent_manual:.4f}")
    print(f"YFinance: {recent_yf:.4f}")
    print(f"Difference: {diff_percent:.4f}%")

Calculating manually adjusted close prices...
Processing EQNR.OL...
Processing DNB.OL...
Processing AKRBP.OL...
Processing ORK.OL...
Processing MOWI.OL...

Adjusted Close Data Shape: (1257, 5)

First 5 rows of manually calculated adjusted close prices:
                             EQNR.OL     DNB.OL    AKRBP.OL     ORK.OL  \
Date                                                                     
2020-10-15 00:00:00+02:00  90.435293  91.401270  100.573412  68.316482   
2020-10-16 00:00:00+02:00  91.564030  92.778313  102.563961  68.954538   
2020-10-19 00:00:00+02:00  91.358804  95.222568  102.563961  68.895182   
2020-10-20 00:00:00+02:00  90.366891  95.360270  101.214435  68.924860   
2020-10-21 00:00:00+02:00  90.024851  96.393053  101.214435  68.405509   

                              MOWI.OL  
Date                                   
2020-10-15 00:00:00+02:00  144.039953  
2020-10-16 00:00:00+02:00  144.039953  
2020-10-19 00:00:00+02:00  145.837782  
2020-10-20 00:00:00+02:00  1

In [10]:
# Create comparison chart: Regular Close vs Adjusted Close
from plotly.subplots import make_subplots

# Select one ticker for detailed comparison (you can change this)
sample_ticker = "EQNR.OL"

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=(f'{sample_ticker} - Regular Close Price', f'{sample_ticker} - Adjusted Close Price'),
    vertical_spacing=0.1
)

# Regular close price
fig.add_trace(
    go.Scatter(
        x=close_prices.index,
        y=close_prices[sample_ticker],
        mode='lines',
        name='Regular Close',
        line=dict(color='blue', width=2)
    ),
    row=1, col=1
)

# Adjusted close price
fig.add_trace(
    go.Scatter(
        x=adjusted_close_prices.index,
        y=adjusted_close_prices[sample_ticker],
        mode='lines',
        name='Adjusted Close',
        line=dict(color='red', width=2)
    ),
    row=2, col=1
)

fig.update_layout(
    title=f'Sammenligning: Vanlig vs Justert Sluttkurs - {sample_ticker}',
    height=800,
    showlegend=True
)

fig.update_xaxes(title_text="Dato", row=2, col=1)
fig.update_yaxes(title_text="Pris (NOK)", row=1, col=1)
fig.update_yaxes(title_text="Justert Pris (NOK)", row=2, col=1)

fig.show()

# Show the difference in actual numbers
print(f"\n{sample_ticker} - Price Comparison (Last 10 days):")
comparison_df = pd.DataFrame({
    'Date': close_prices.index[-10:],
    'Regular Close': close_prices[sample_ticker].iloc[-10:].values,
    'Adjusted Close': adjusted_close_prices[sample_ticker].iloc[-10:].values
})
comparison_df['Difference'] = comparison_df['Regular Close'] - comparison_df['Adjusted Close']
comparison_df['Difference %'] = ((comparison_df['Regular Close'] - comparison_df['Adjusted Close']) / comparison_df['Regular Close'] * 100).round(4)

print(comparison_df.to_string(index=False))


EQNR.OL - Price Comparison (Last 10 days):
                     Date  Regular Close  Adjusted Close  Difference  Difference %
2025-10-01 00:00:00+02:00     246.000000      246.000000         0.0           0.0
2025-10-02 00:00:00+02:00     246.399994      246.399994         0.0           0.0
2025-10-03 00:00:00+02:00     247.600006      247.600006         0.0           0.0
2025-10-06 00:00:00+02:00     253.199997      253.199997         0.0           0.0
2025-10-07 00:00:00+02:00     249.399994      249.399994         0.0           0.0
2025-10-08 00:00:00+02:00     246.899994      246.899994         0.0           0.0
2025-10-09 00:00:00+02:00     246.100006      246.100006         0.0           0.0
2025-10-10 00:00:00+02:00     239.000000      239.000000         0.0           0.0
2025-10-13 00:00:00+02:00     237.300003      237.300003         0.0           0.0
2025-10-14 00:00:00+02:00     235.199997      235.199997         0.0           0.0


In [12]:
# Create a single plot showing all adjusted close prices
fig = go.Figure()

# Define colors for each ticker
colors = ['#2f4d8e', '#ffa659', '#ff6464', '#79ffbc', '#c180ff']

for i, ticker in enumerate(TICKERS):
    if ticker in adjusted_close_prices.columns:
        fig.add_trace(go.Scatter(
            x=adjusted_close_prices.index,
            y=adjusted_close_prices[ticker],
            mode='lines',
            name=ticker,
            line=dict(color=colors[i], width=2),
            hovertemplate=f'<b>{ticker}</b><br>' +
                         'Dato: %{x}<br>' +
                         'Justert sluttkurs: %{y:.2f} NOK<br>' +
                         '<extra></extra>'
        ))

# Update layout
fig.update_layout(
    title='Historisk justert sluttkurs - Norske Selskaper',
    xaxis_title='Dato',
    yaxis_title='Justert sluttkurs (NOK)',
    hovermode='x unified',
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01,
        bgcolor="rgba(255, 255, 255, 0.8)",
        bordercolor="black",
        borderwidth=1
    ),
    height=600,
    width=1000,
    plot_bgcolor='white',
    xaxis=dict(
        showgrid=True,
        gridcolor='lightgray'
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor='lightgray'
    )
)

# Show the plot
fig.show()

# Print summary statistics
print("\n" + "="*60)
print("ADJUSTED CLOSE PRICE SUMMARY STATISTICS")
print("="*60)
print(adjusted_close_prices.describe().round(2))


ADJUSTED CLOSE PRICE SUMMARY STATISTICS
       EQNR.OL   DNB.OL  AKRBP.OL   ORK.OL  MOWI.OL
count  1257.00  1257.00   1257.00  1257.00  1257.00
mean    219.93   174.80    217.37    72.79   182.12
std      58.13    43.98     35.37    16.56    20.36
min      82.57    87.75     96.09    53.99   119.63
25%     174.45   145.73    200.41    62.20   170.67
50%     243.25   161.36    225.20    65.87   182.89
75%     259.33   199.21    241.43    81.55   195.37
max     308.89   284.50    305.46   118.40   234.28


In [14]:
# Calculate and visualize standard deviation (risk metrics)
from plotly.subplots import make_subplots

# Calculate daily returns for adjusted close prices
returns = adjusted_close_prices.pct_change().dropna()

# Calculate risk metrics
risk_metrics = []
for ticker in TICKERS:
    if ticker in returns.columns:
        daily_std = returns[ticker].std()
        annual_std = daily_std * (252**0.5)  # Annualized (252 trading days)
        risk_metrics.append({
            'Ticker': ticker,
            'Daily_Std': daily_std * 100,  # Convert to percentage
            'Annual_Std': annual_std * 100  # Convert to percentage
        })

risk_df = pd.DataFrame(risk_metrics)

# Create subplot with two bar charts
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Daglig standardavvik (%)', 'Årlig standardavvik (%)'),
    horizontal_spacing=0.1
)

# Same colors as before
colors = ['#2f4d8e', '#ffa659', '#ff6464', '#79ffbc', '#c180ff']

# Daily standard deviation with values on bars
fig.add_trace(
    go.Bar(
        x=risk_df['Ticker'],
        y=risk_df['Daily_Std'],
        name='Daglig std',
        marker_color=colors,
        text=[f'{val:.4f}%' for val in risk_df['Daily_Std']],
        textposition='outside',
        textfont=dict(size=10, color='black'),
        hovertemplate='<b>%{x}</b><br>' +
                     'Daglig std: %{y:.4f}%<br>' +
                     '<extra></extra>'
    ),
    row=1, col=1
)

# Annual standard deviation with values on bars
fig.add_trace(
    go.Bar(
        x=risk_df['Ticker'],
        y=risk_df['Annual_Std'],
        name='Årlig std',
        marker_color=colors,
        text=[f'{val:.2f}%' for val in risk_df['Annual_Std']],
        textposition='outside',
        textfont=dict(size=10, color='black'),
        hovertemplate='<b>%{x}</b><br>' +
                     'Årlig std: %{y:.2f}%<br>' +
                     '<extra></extra>'
    ),
    row=1, col=2
)

# Update layout
fig.update_layout(
    title='Risikoanalyse: Standardavvik for norske aksjer',
    height=550,  # Increased height to accommodate text labels
    width=1000,
    showlegend=False,
    plot_bgcolor='white'
)

# Update axes - add some top margin for the text labels
fig.update_yaxes(
    title_text="Standardavvik (%)", 
    row=1, col=1, 
    showgrid=True, 
    gridcolor='lightgray',
    range=[0, max(risk_df['Daily_Std']) * 1.15]  # Add 15% padding at top
)
fig.update_yaxes(
    title_text="Standardavvik (%)", 
    row=1, col=2, 
    showgrid=True, 
    gridcolor='lightgray',
    range=[0, max(risk_df['Annual_Std']) * 1.15]  # Add 15% padding at top
)
fig.update_xaxes(row=1, col=1, showgrid=False)
fig.update_xaxes(row=1, col=2, showgrid=False)

fig.show()

# Print the risk metrics table
print("\n" + "="*60)
print("RISIKOANALYSE - STANDARDAVVIK")
print("="*60)
print("Standardavvik måler volatilitet/risiko. Høyere verdier indikerer mer risiko.")
print("\nRisikometrikker for alle aksjer:")
print("-" * 50)

for _, row in risk_df.iterrows():
    print(f"{row['Ticker']:10} | Daglig: {row['Daily_Std']:7.4f}% | Årlig: {row['Annual_Std']:6.2f}%")

print("\nRanking etter risiko (årlig standardavvik):")
print("-" * 40)
risk_sorted = risk_df.sort_values('Annual_Std', ascending=False)
for i, (_, row) in enumerate(risk_sorted.iterrows(), 1):
    risk_level = "Høy" if row['Annual_Std'] > 30 else "Middels" if row['Annual_Std'] > 20 else "Lav"
    print(f"{i}. {row['Ticker']:10} | {row['Annual_Std']:6.2f}% | Risiko: {risk_level}")

# Calculate correlation matrix
print(f"\n{'='*60}")
print("KORRELASJONSMATRISE (Adjusted Close Returns)")
print("="*60)
correlation_matrix = returns.corr()
print(correlation_matrix.round(3))


RISIKOANALYSE - STANDARDAVVIK
Standardavvik måler volatilitet/risiko. Høyere verdier indikerer mer risiko.

Risikometrikker for alle aksjer:
--------------------------------------------------
EQNR.OL    | Daglig:  1.9312% | Årlig:  30.66%
DNB.OL     | Daglig:  1.4141% | Årlig:  22.45%
AKRBP.OL   | Daglig:  2.1708% | Årlig:  34.46%
ORK.OL     | Daglig:  1.1920% | Årlig:  18.92%
MOWI.OL    | Daglig:  1.6338% | Årlig:  25.94%

Ranking etter risiko (årlig standardavvik):
----------------------------------------
1. AKRBP.OL   |  34.46% | Risiko: Høy
2. EQNR.OL    |  30.66% | Risiko: Høy
3. MOWI.OL    |  25.94% | Risiko: Middels
4. DNB.OL     |  22.45% | Risiko: Middels
5. ORK.OL     |  18.92% | Risiko: Lav

KORRELASJONSMATRISE (Adjusted Close Returns)
          EQNR.OL  DNB.OL  AKRBP.OL  ORK.OL  MOWI.OL
EQNR.OL     1.000   0.249     0.750  -0.004    0.155
DNB.OL      0.249   1.000     0.272   0.119    0.310
AKRBP.OL    0.750   0.272     1.000  -0.042    0.176
ORK.OL     -0.004   0.119    -